In [24]:
!pip install google-genai youtube-transcript-api
!pip install --upgrade youtube-transcript-api


In [25]:
from google.colab import userdata
GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')


In [26]:
from youtube_transcript_api import YouTubeTranscriptApi

def get_transcript(video_id, languages=['en']):
    transcript = YouTubeTranscriptApi().fetch(video_id, languages=languages)
    print(transcript)

    text = " ".join([item.text for item in transcript])
    return text

# Correct usage
video_id = "nLj-DGIC_xM"
print(get_transcript(video_id))

FetchedTranscript(snippets=[FetchedTranscriptSnippet(text='Hello everyone, my name is Priya and I', start=0.8, duration=4.88), FetchedTranscriptSnippet(text='welcome all of you to the session. In', start=3.28, duration=4.32), FetchedTranscriptSnippet(text='this session, we are going to discuss', start=5.68, duration=5.28), FetchedTranscriptSnippet(text='about data types in JavaScript. So these', start=7.6, duration=5.52), FetchedTranscriptSnippet(text='data types are divided into two', start=10.96, duration=4.799), FetchedTranscriptSnippet(text='different categories. One is primitive', start=13.12, duration=6.239), FetchedTranscriptSnippet(text='types and next is reference type. So', start=15.759, duration=5.36), FetchedTranscriptSnippet(text='first we are going to discuss about', start=19.359, duration=4.561), FetchedTranscriptSnippet(text='primitive types in JavaScript. Then we', start=21.119, duration=4.721), FetchedTranscriptSnippet(text='are going to move towards reference', start

In [27]:
from google import genai

client = genai.Client(api_key=GOOGLE_API_KEY)


In [28]:
import json
import re

def analyze_video(transcript):
    prompt = f"""
Analyze this YouTube video transcript and return a structured breakdown.

Transcript:
{transcript}

Return in JSON format with these exact keys:
- summary (2-3 sentence short summary)
- detailed_explanation (a longer paragraph explaining the video's content)
- key_points (list of short bullet points)
- insights (list of short bullet points, important takeaways)
- qna (list of objects, each with "question" and "answer" keys — exactly 5 items)
- topics (list of up to 6 short topic/keyword tags for this video)

Rules:
- Output ONLY JSON
- No extra text, no markdown code fences
"""
    response = client.models.generate_content(
        model="gemini-3.5-flash",
        contents=prompt
    )
    return response.text

In [29]:
video_id = "nLj-DGIC_xM"  # replace
transcript = get_transcript(video_id)

result = analyze_video(transcript)
print(result)



FetchedTranscript(snippets=[FetchedTranscriptSnippet(text='Hello everyone, my name is Priya and I', start=0.8, duration=4.88), FetchedTranscriptSnippet(text='welcome all of you to the session. In', start=3.28, duration=4.32), FetchedTranscriptSnippet(text='this session, we are going to discuss', start=5.68, duration=5.28), FetchedTranscriptSnippet(text='about data types in JavaScript. So these', start=7.6, duration=5.52), FetchedTranscriptSnippet(text='data types are divided into two', start=10.96, duration=4.799), FetchedTranscriptSnippet(text='different categories. One is primitive', start=13.12, duration=6.239), FetchedTranscriptSnippet(text='types and next is reference type. So', start=15.759, duration=5.36), FetchedTranscriptSnippet(text='first we are going to discuss about', start=19.359, duration=4.561), FetchedTranscriptSnippet(text='primitive types in JavaScript. Then we', start=21.119, duration=4.721), FetchedTranscriptSnippet(text='are going to move towards reference', start

In [30]:
def ask_question(transcript, question):
   prompt = f"""
   Answer the question based on this video transcript:

   Transcript:
   {transcript}

   Question:
   {question}
   """

   response = client.models.generate_content(
       model="gemini-3.5-flash",
       contents=prompt
   )

   return response.text
# Example
print(ask_question(transcript, "What is the main topic?"))



Based on the transcript, the main topic of the session is **JavaScript data types**, specifically divided into:

1. **Primitive types** (such as numbers, strings, booleans, null, and undefined).
2. **Reference types** (such as arrays and objects, and how they are stored in heap memory).

Additionally, the session covers **conditional control constructs (if/else statements)** and demonstrates how to apply these concepts by building a practical **counter application** in JavaScript.


In [31]:
from IPython.display import HTML, display

video_id = "nLj-DGIC_xM"  # replace with your video ID
transcript = get_transcript(video_id)
raw_result = analyze_video(transcript)

# Strip markdown fences if Gemini adds them anyway
cleaned = re.sub(r"^```json|```$", "", raw_result.strip(), flags=re.MULTILINE).strip()
data = json.loads(cleaned)

summary = data.get("summary", "")
detailed = data.get("detailed_explanation", "")
key_points = data.get("key_points", [])
insights = data.get("insights", [])
qna = data.get("qna", [])
topics = data.get("topics", [])

topics_html = "".join(
    f'<span style="background:#e0e7ff; color:#4338ca; padding:5px 12px; border-radius:14px; '
    f'font-size:13px; font-weight:500; margin:4px; display:inline-block;">{t}</span>'
    for t in topics
)
key_points_html = "".join(
    f'<li style="margin-bottom:8px;"><span style="color:#4338ca;">▸</span> {k}</li>' for k in key_points
)
insights_html = "".join(
    f'<li style="margin-bottom:8px;"><span style="color:#f59e0b;">💡</span> {i}</li>' for i in insights
)
qna_html = "".join(
    f'''<div style="margin-bottom:14px;">
        <div style="font-weight:600; color:#111827; font-size:13px;">Q: {qa.get("question","")}</div>
        <div style="color:#374151; font-size:13px; margin-top:4px;">A: {qa.get("answer","")}</div>
    </div>'''
    for qa in qna
)

dashboard = f"""
<div style="font-family:'Segoe UI', Arial, sans-serif; max-width:900px; margin:auto;
            border-radius:14px; overflow:hidden; box-shadow:0 6px 20px rgba(0,0,0,0.12);">

  <div style="background:#312e81; color:white; text-align:center; padding:16px; font-size:18px; font-weight:600;">
    🎥 AI YouTube Video Summarizer — Dashboard
  </div>

  <div style="background:#f8f9fc; padding:20px; display:grid; grid-template-columns:1fr 1fr; gap:16px;">

    <!-- Summary -->
    <div style="background:white; border-radius:12px; padding:18px; box-shadow:0 1px 4px rgba(0,0,0,0.06); grid-column:1 / -1;">
      <div style="font-weight:600; color:#111827; margin-bottom:8px;">📌 Summary</div>
      <div style="font-size:14px; color:#374151; line-height:1.6;">{summary}</div>
    </div>

    <!-- Topics -->
    <div style="background:white; border-radius:12px; padding:18px; box-shadow:0 1px 4px rgba(0,0,0,0.06); grid-column:1 / -1;">
      <div style="font-weight:600; color:#111827; margin-bottom:10px;">🏷️ Topics Covered</div>
      <div>{topics_html}</div>
    </div>

    <!-- Detailed Explanation -->
    <div style="background:white; border-radius:12px; padding:18px; box-shadow:0 1px 4px rgba(0,0,0,0.06); grid-column:1 / -1;">
      <div style="font-weight:600; color:#111827; margin-bottom:8px;">📖 Detailed Explanation</div>
      <div style="font-size:13px; color:#374151; line-height:1.6;">{detailed}</div>
    </div>

    <!-- Key Points -->
    <div style="background:white; border-radius:12px; padding:18px; box-shadow:0 1px 4px rgba(0,0,0,0.06);">
      <div style="font-weight:600; color:#111827; margin-bottom:10px;">🔑 Key Points</div>
      <ul style="list-style:none; padding:0; margin:0; font-size:13px; color:#374151;">{key_points_html}</ul>
    </div>

    <!-- Insights -->
    <div style="background:white; border-radius:12px; padding:18px; box-shadow:0 1px 4px rgba(0,0,0,0.06);">
      <div style="font-weight:600; color:#111827; margin-bottom:10px;">💡 Important Insights</div>
      <ul style="list-style:none; padding:0; margin:0; font-size:13px; color:#374151;">{insights_html}</ul>
    </div>

    <!-- Q&A -->
    <div style="background:white; border-radius:12px; padding:18px; box-shadow:0 1px 4px rgba(0,0,0,0.06); grid-column:1 / -1;">
      <div style="font-weight:600; color:#111827; margin-bottom:12px;">❓ Questions & Answers</div>
      {qna_html}
    </div>

  </div>
</div>
"""

display(HTML(dashboard))

FetchedTranscript(snippets=[FetchedTranscriptSnippet(text='Hello everyone, my name is Priya and I', start=0.8, duration=4.88), FetchedTranscriptSnippet(text='welcome all of you to the session. In', start=3.28, duration=4.32), FetchedTranscriptSnippet(text='this session, we are going to discuss', start=5.68, duration=5.28), FetchedTranscriptSnippet(text='about data types in JavaScript. So these', start=7.6, duration=5.52), FetchedTranscriptSnippet(text='data types are divided into two', start=10.96, duration=4.799), FetchedTranscriptSnippet(text='different categories. One is primitive', start=13.12, duration=6.239), FetchedTranscriptSnippet(text='types and next is reference type. So', start=15.759, duration=5.36), FetchedTranscriptSnippet(text='first we are going to discuss about', start=19.359, duration=4.561), FetchedTranscriptSnippet(text='primitive types in JavaScript. Then we', start=21.119, duration=4.721), FetchedTranscriptSnippet(text='are going to move towards reference', start